In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
TRAIN_DIR = '/content/drive/MyDrive/INTERNSHIP/PlantVillage/train'
VAL_DIR   = '/content/drive/MyDrive/INTERNSHIP/PlantVillage/val'

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
import pickle

In [ ]:
# define training transformations
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# define validation transformations
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# load datasets using imagefolder
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
val_dataset   = datasets.ImageFolder(root=VAL_DIR, transform=val_transforms)

In [ ]:
# print class names
class_names = train_dataset.classes

print("total classes:", len(class_names))
for idx, cls in enumerate(class_names):
    print(idx, ":", cls)

total classes: 38
0 : Apple___Apple_scab
1 : Apple___Black_rot
2 : Apple___Cedar_apple_rust
3 : Apple___healthy
4 : Blueberry___healthy
5 : Cherry_(including_sour)___Powdery_mildew
6 : Cherry_(including_sour)___healthy
7 : Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
8 : Corn_(maize)___Common_rust_
9 : Corn_(maize)___Northern_Leaf_Blight
10 : Corn_(maize)___healthy
11 : Grape___Black_rot
12 : Grape___Esca_(Black_Measles)
13 : Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
14 : Grape___healthy
15 : Orange___Haunglongbing_(Citrus_greening)
16 : Peach___Bacterial_spot
17 : Peach___healthy
18 : Pepper,_bell___Bacterial_spot
19 : Pepper,_bell___healthy
20 : Potato___Early_blight
21 : Potato___Late_blight
22 : Potato___healthy
23 : Raspberry___healthy
24 : Soybean___healthy
25 : Squash___Powdery_mildew
26 : Strawberry___Leaf_scorch
27 : Strawberry___healthy
28 : Tomato___Bacterial_spot
29 : Tomato___Early_blight
30 : Tomato___Late_blight
31 : Tomato___Leaf_Mold
32 : Tomato___Septoria_leaf_

In [ ]:
# calculate class imbalance weights
targets = train_dataset.targets
class_counts = np.bincount(targets)

class_weights = 1. / class_counts
sample_weights = class_weights[targets]

In [ ]:
# create weighted sampler
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

In [ ]:
# create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
# check one batch
images, labels = next(iter(train_loader))

print("images shape:", images.shape)
print("labels shape:", labels.shape)

images shape: torch.Size([32, 3, 224, 224])
labels shape: torch.Size([32])


In [ ]:
# save class names for future inference
save_path = "/content/drive/MyDrive/INTERNSHIP/image_class_names.pkl"

with open(save_path, "wb") as f:
    pickle.dump(class_names, f)

print("class names saved successfully!")
print("saved at:", save_path)

class names saved successfully!
saved at: /content/drive/MyDrive/INTERNSHIP/image_class_names.pkl
